# Lego Avatar Pipeline — landmarks → embeddings → vector search

Implementation of the architecture from the research doc (§5):

1. **MediaPipe Face Mesh** isolates each facial region from the user photo.
2. **ArcFace** (geometric facial regions) + **CLIP** (appearance regions) turn every crop into a vector.
3. **FAISS** finds the nearest piece *within its own category*.

### Two things this notebook adds to the doc, and why

**A. The domain gap is the main technical risk, not the encoder choice.**
The query is a photograph of a real eye; the bank is a *drawn* eye. ArcFace and CLIP were trained on
photographs, so photo→drawing cosine distance is largely dominated by "photo vs. cartoon", not by
"round vs. almond". Cell 6 therefore supports a **reference set**: real photos labelled with the piece
they correspond to. Matching then runs photo→photo (query → reference), and the piece is read off the
winning reference's label. This removes the gap entirely and is why you were asked for the real-people
images. Pure photo→piece matching is kept as a fallback (`MATCH_MODE = "direct"`) so you can measure
the difference rather than take my word for it.

**B. ArcFace on an isolated crop is off-distribution.**
ArcFace's embedding space is trained on *whole aligned faces* at 112×112, with a loss that deliberately
discards everything except identity. Feeding it a cropped eye is untested territory — it may work, it may
collapse. Cell 10 A/B-tests ArcFace vs CLIP per category on your own data and prints top-1/top-3 accuracy,
so the routing table in cell 7 is a measured decision, not an assumption.

**C. For skin tone, do not use a neural embedding.**
Skin tone is a 3-number problem. Median CIELAB of the cheek patch, compared with ΔE2000, is more accurate,
100× cheaper and fully interpretable. Cell 7 routes `skin_tone` to a colour metric, not to CLIP.

## Cell 1 — Runtime & install

Set **Runtime → Change runtime type → T4 GPU** first. CPU works but cell 8 (indexing the bank) will be
slow if the bank is large.

Versions are pinned because `insightface` + `onnxruntime` + `numpy` break against each other often;
unpinned installs are the #1 cause of this notebook failing on a fresh Colab.

In [ ]:
# insightface ships the ArcFace 'buffalo_l' pack (detector + recognition ONNX models).
# onnxruntime-gpu is what actually executes them; the CPU wheel is the fallback.
# open_clip_torch is used instead of the HF CLIP wrapper because it exposes the raw image tower
#   and lets us swap checkpoints (ViT-B/32 -> ViT-L/14) with one string.
# faiss-cpu is enough here: exact search over a few thousand pieces is sub-millisecond.
#   Only move to faiss-gpu / IVF-PQ past ~1M vectors, which a piece bank will never reach.
!pip install -q insightface==0.7.3 onnxruntime-gpu==1.18.0 open_clip_torch==2.24.0 \
                faiss-cpu==1.8.0 mediapipe==0.10.14 opencv-python-headless==4.10.0.84 \
                numpy==1.26.4 scikit-image==0.24.0

import importlib, sys
for m in ["insightface", "open_clip", "faiss", "mediapipe", "cv2", "skimage"]:
    importlib.import_module(m)
print("all imports OK —", sys.version.split()[0])

## Cell 2 — Data contract

Upload your assets to Google Drive and point `ROOT` at them. The pipeline expects this layout:

```
ROOT/
├── pieces/                     # the Lego bank — the things we retrieve
│   ├── eyes/eye_001.png
│   ├── eyebrows/brow_014.png
│   ├── nose/…  mouth/…  face_shape/…  hair/…  beard/…
├── refs/                       # OPTIONAL but strongly recommended (see intro, point A)
│   ├── eyes/eye_001/person_a.jpg     # real photos of people whose eyes == piece eye_001
│   ├── eyes/eye_001/person_b.jpg
│   └── eyes/eye_007/person_c.jpg
└── queries/                    # full-face photos to run the pipeline on
    └── test_01.jpg
```

The `refs/<category>/<piece_id>/` folder name **is** the label. That is the whole labelling scheme —
no CSV, no annotation tool. Adding a piece = dropping a PNG in `pieces/` and (ideally) a few real photos
in `refs/`; nothing is retrained, the index is just rebuilt.

`refs` images can be **full faces** — cell 5 crops them the same way it crops a query, so the reference
embedding and the query embedding come from an identical transform. That symmetry is the point.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/lego_avatar')   # <-- EDIT ME
PIECES_DIR  = ROOT / 'pieces'
REFS_DIR    = ROOT / 'refs'
QUERIES_DIR = ROOT / 'queries'
CACHE_DIR   = ROOT / '_cache'          # embeddings + FAISS indices are cached here
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Order matters only for the compositing preview (cell 11): later = drawn on top.
CATEGORIES = ["face_shape", "skin_tone", "eyes", "eyebrows", "nose", "mouth", "beard", "hair"]

for d in (PIECES_DIR, REFS_DIR, QUERIES_DIR):
    print(f"{'OK ' if d.exists() else 'MISSING '} {d}")

IMG_EXT = {'.png', '.jpg', '.jpeg', '.webp'}

def scan_bank():
    # Returns {category: [(piece_id, path), ...]}. piece_id is the filename stem and is the
    # join key between pieces/ and refs/ — keep them identical or refs will silently orphan.
    bank = {}
    for cat in CATEGORIES:
        d = PIECES_DIR / cat
        items = sorted((p.stem, p) for p in d.glob('*') if p.suffix.lower() in IMG_EXT) if d.exists() else []
        bank[cat] = items
    return bank

def scan_refs():
    # Returns {category: [(piece_id, path), ...]} where piece_id comes from the PARENT folder name.
    refs = {}
    for cat in CATEGORIES:
        d = REFS_DIR / cat
        items = []
        if d.exists():
            for piece_dir in sorted(d.iterdir()):
                if piece_dir.is_dir():
                    items += [(piece_dir.name, p) for p in sorted(piece_dir.glob('*'))
                              if p.suffix.lower() in IMG_EXT]
        refs[cat] = items
    return refs

BANK, REFS = scan_bank(), scan_refs()
print(f"\n{'category':<12}{'pieces':>8}{'refs':>8}{'refs/piece':>12}")
for cat in CATEGORIES:
    n_p, n_r = len(BANK[cat]), len(REFS[cat])
    ids = len({i for i, _ in REFS[cat]})
    print(f"{cat:<12}{n_p:>8}{n_r:>8}{(n_r/ids if ids else 0):>12.1f}")

# Orphan check — a ref pointing at a piece_id that does not exist is a silent accuracy killer,
# because it can win a search and then fail to composite.
for cat in CATEGORIES:
    piece_ids = {i for i, _ in BANK[cat]}
    orphans = {i for i, _ in REFS[cat]} - piece_ids
    if orphans:
        print(f"WARNING {cat}: refs reference unknown piece_ids -> {sorted(orphans)[:5]}")

## Cell 3 — Face Mesh landmarks

The doc proposed a 68-point model (dlib / TCDCN). This uses MediaPipe Face Mesh — **468 points**, and the
reason is not "more is better", it is that 68-point models give you 6 points per eye and none for the hair
line or cheek interior. 468 points give dense contours you can crop from directly, plus MediaPipe is
already CPU-real-time and needs no separate download.

`refine_landmarks=True` adds iris points (468→478). Costs ~1 ms and gives an eye-scale reference that is
independent of head size — used below to normalise crop padding.

In [ ]:
import cv2, numpy as np, mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
_FACE_MESH = mp_face_mesh.FaceMesh(
    static_image_mode=True,       # every input is an independent photo, no temporal tracking
    max_num_faces=1,              # multi-face is out of scope; ambiguous input should fail loudly
    refine_landmarks=True,        # +iris points
    min_detection_confidence=0.5,
)

def load_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def get_landmarks(rgb):
    # Returns (478, 2) float array in PIXEL coordinates, or None if no face was found.
    res = _FACE_MESH.process(rgb)
    if not res.multi_face_landmarks:
        return None
    h, w = rgb.shape[:2]
    lm = res.multi_face_landmarks[0].landmark
    return np.array([[p.x * w, p.y * h] for p in lm], dtype=np.float32)

# Landmark index groups. These are stable MediaPipe canonical indices.
REGION_IDX = {
    "eyes":       [33,133,160,159,158,157,173,155,154,153,145,144,163,7,          # right eye ring
                   362,263,387,386,385,384,398,382,381,380,374,373,390,249],      # left eye ring
    "eyebrows":   [70,63,105,66,107,55,65,52,53,46,
                   300,293,334,296,336,285,295,282,283,276],
    "nose":       [168,6,197,195,5,4,1,19,94,2,98,97,326,327,115,344,220,440],
    "mouth":      [61,291,0,17,13,14,78,308,37,267,84,314,181,405,146,375],
    "face_shape": list(range(0, 17)) + [234,454,10,152,127,356,132,361,172,397],
    "beard":      [61,291,17,18,200,199,175,152,148,176,149,150,136,172,58,132,
                   377,400,378,379,365,397,288,361],
    "hair":       [10,109,338,67,297,54,284,21,251,103,332],   # forehead/hairline anchor row
    "skin_tone":  [50,280,205,425,101,330,116,345],            # cheek interior, avoids lips/brows/eyes
}
print("regions:", list(REGION_IDX))

## Cell 4 — Region crops

Two decisions worth defending:

- **Crops are square and padded relative to inter-ocular distance, not to image size.** A 20 px pad means
  something different on a selfie and on a 4K portrait; `0.35 × IOD` means the same thing on both. This is
  what makes a query crop and a reference crop comparable at all.
- **Crops are taken from a roll-aligned image.** The face is rotated so the eye line is horizontal before
  cropping, so a tilted head does not produce a tilted eye crop that the encoder reads as a different shape.

`hair` and `beard` are deliberately *offset* regions rather than landmark hulls: hair sits **above** the
hairline landmarks and beard **below** the lip line, and neither has landmarks of its own.

In [ ]:
from typing import Optional

L_EYE_OUT, R_EYE_OUT = 263, 33     # outer eye corners, used for roll alignment + IOD scale

def align_face(rgb, lms):
    # Rotate about the eye-line midpoint so the two outer eye corners are horizontal.
    # Returns (aligned_image, aligned_landmarks).
    p_l, p_r = lms[L_EYE_OUT], lms[R_EYE_OUT]
    dy, dx = (p_l - p_r)[1], (p_l - p_r)[0]
    angle = np.degrees(np.arctan2(dy, dx))
    center = tuple(((p_l + p_r) / 2).tolist())
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    h, w = rgb.shape[:2]
    out = cv2.warpAffine(rgb, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    lms_h = np.hstack([lms, np.ones((len(lms), 1), np.float32)])
    return out, (lms_h @ M.T).astype(np.float32)

def iod(lms):
    # Inter-ocular distance = the scale unit for every pad below.
    return float(np.linalg.norm(lms[L_EYE_OUT] - lms[R_EYE_OUT]))

# pad = fraction of IOD added around the region bbox; dy = vertical shift in IOD units
# (negative = up). hair/beard use dy because their content is not where their landmarks are.
CROP_CFG = {
    "eyes":       dict(pad=0.30, dy=0.00),
    "eyebrows":   dict(pad=0.35, dy=0.00),
    "nose":       dict(pad=0.25, dy=0.00),
    "mouth":      dict(pad=0.30, dy=0.00),
    "face_shape": dict(pad=0.10, dy=0.00),
    "beard":      dict(pad=0.20, dy=0.10),
    "hair":       dict(pad=0.90, dy=-0.75),
    "skin_tone":  dict(pad=0.15, dy=0.00),
}

def crop_region(rgb_aligned, lms_aligned, category, out_size=224) -> Optional[np.ndarray]:
    idx = REGION_IDX[category]
    pts = lms_aligned[idx]
    cfg = CROP_CFG[category]
    s = iod(lms_aligned)

    x0, y0 = pts.min(0); x1, y1 = pts.max(0)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2 + cfg["dy"] * s
    half = max(x1 - x0, y1 - y0) / 2 + cfg["pad"] * s   # square box -> no aspect distortion on resize

    h, w = rgb_aligned.shape[:2]
    X0, Y0 = int(round(cx - half)), int(round(cy - half))
    X1, Y1 = int(round(cx + half)), int(round(cy + half))

    # Pad instead of clamping: clamping would change the crop's aspect and shift the region
    # off-centre, which is exactly the kind of silent inconsistency that poisons a retrieval index.
    pl, pt = max(0, -X0), max(0, -Y0)
    pr, pb = max(0, X1 - w), max(0, Y1 - h)
    if pl or pt or pr or pb:
        rgb_aligned = cv2.copyMakeBorder(rgb_aligned, pt, pb, pl, pr, cv2.BORDER_REPLICATE)
        X0 += pl; X1 += pl; Y0 += pt; Y1 += pt

    patch = rgb_aligned[Y0:Y1, X0:X1]
    if patch.size == 0:
        return None
    return cv2.resize(patch, (out_size, out_size), interpolation=cv2.INTER_CUBIC)

def crops_from_photo(path, categories=None, out_size=224):
    # Full photo -> {category: crop}. Returns {} when no face is detected, so callers can
    # count and report failures rather than silently indexing garbage.
    rgb = load_rgb(path)
    lms = get_landmarks(rgb)
    if lms is None:
        return {}
    a_rgb, a_lms = align_face(rgb, lms)
    cats = categories or CATEGORIES
    out = {}
    for c in cats:
        p = crop_region(a_rgb, a_lms, c, out_size)
        if p is not None:
            out[c] = p
    return out

## Cell 5 — Visual check before anything expensive

Run this on 3–5 of your own photos **before** building any index. If the crops are wrong, every number
downstream is meaningless and you will waste an hour debugging FAISS for a cropping bug. The `hair` and
`beard` offsets in `CROP_CFG` are the two you will most likely need to tune for your photo framing.

In [ ]:
import matplotlib.pyplot as plt

def preview_crops(path):
    crops = crops_from_photo(path)
    if not crops:
        print(f"NO FACE DETECTED: {path}"); return
    fig, axes = plt.subplots(1, len(crops), figsize=(2.0 * len(crops), 2.4))
    axes = np.atleast_1d(axes)
    for ax, (c, img) in zip(axes, crops.items()):
        ax.imshow(img); ax.set_title(c, fontsize=9); ax.axis('off')
    fig.suptitle(Path(path).name, fontsize=10); plt.tight_layout(); plt.show()

for p in sorted(QUERIES_DIR.glob('*'))[:3]:
    if p.suffix.lower() in IMG_EXT:
        preview_crops(p)

## Cell 6 — Encoders: ArcFace, CLIP, and a colour metric

**ArcFace** — loaded straight from the `buffalo_l` recognition head, bypassing insightface's `FaceAnalysis`
wrapper because that wrapper insists on re-detecting a face inside the image; our input is already a crop of
one region, and no detector will find a face in a picture of an eyebrow. We feed the crop through the same
112×112 / BGR / `(x-127.5)/127.5` preprocessing the model was trained with, then L2-normalise the 512-d output.

**CLIP** — ViT-B/32 image tower, 512-d, also L2-normalised. Used for regions where the discriminating signal
is texture and colour (hair, beard) rather than geometry.

**Skin tone** — no network. Median CIELAB over the cheek patch, matched by ΔE2000. Stated plainly: a
512-d embedding of a cheek is a worse skin-tone descriptor than 3 numbers in a perceptually uniform colour
space, and it cannot be inspected when it goes wrong.

Both embeddings are L2-normalised so cosine similarity == inner product, which lets FAISS `IndexFlatIP`
give exact cosine ranking with no extra maths.

In [ ]:
import torch, open_clip, insightface
from insightface.model_zoo import get_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# ---- ArcFace ----------------------------------------------------------------
from insightface.app import FaceAnalysis
_ = FaceAnalysis(name='buffalo_l')          # side effect: downloads the model pack into ~/.insightface
_arc_path = str(Path.home() / '.insightface/models/buffalo_l/w600k_r50.onnx')
_ARC = get_model(_arc_path)
_ARC.prepare(ctx_id=0 if DEVICE == "cuda" else -1)

def embed_arcface(crops_rgb):
    # crops_rgb: list of HxWx3 uint8 RGB -> (N, 512) float32, L2-normalised.
    batch = []
    for c in crops_rgb:
        img = cv2.resize(c, (112, 112), interpolation=cv2.INTER_CUBIC)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)              # model was trained on BGR
        img = (img.astype(np.float32) - 127.5) / 127.5
        batch.append(img.transpose(2, 0, 1))                     # HWC -> CHW
    blob = np.stack(batch).astype(np.float32)
    feats = _ARC.session.run(None, {_ARC.session.get_inputs()[0].name: blob})[0]
    return feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-9)

# ---- CLIP -------------------------------------------------------------------
_clip_model, _, _clip_pre = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k')
_clip_model = _clip_model.to(DEVICE).eval()

from PIL import Image
@torch.no_grad()
def embed_clip(crops_rgb, batch_size=64):
    out = []
    for i in range(0, len(crops_rgb), batch_size):
        t = torch.stack([_clip_pre(Image.fromarray(c)) for c in crops_rgb[i:i+batch_size]]).to(DEVICE)
        f = _clip_model.encode_image(t).float()
        out.append((f / f.norm(dim=-1, keepdim=True)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

# ---- Skin tone --------------------------------------------------------------
from skimage import color as skcolor

def embed_skin(crops_rgb):
    # Median (not mean) CIELAB: the median is robust to the stray shadow, mole or specular
    # highlight that a mean would smear across the whole estimate.
    out = []
    for c in crops_rgb:
        lab = skcolor.rgb2lab(c.astype(np.float32) / 255.0)
        out.append(np.median(lab.reshape(-1, 3), axis=0))
    return np.asarray(out, dtype=np.float32)     # NOT normalised — this is a metric space, not cosine

def skin_distance(q, bank):
    # CIEDE2000 — the standard perceptual colour difference. ~1.0 = just-noticeable difference.
    from skimage.color import deltaE_ciede2000
    return np.array([deltaE_ciede2000(q, b) for b in bank], dtype=np.float32)

print("encoders ready")

## Cell 7 — Routing table and match mode

`ENCODER` is the doc's §5.3 split made explicit. **Do not trust it yet** — cell 10 measures it on your data
and tells you which assignments to flip.

`MATCH_MODE`:

| mode | query is compared against | needs refs | expected behaviour |
|---|---|---|---|
| `reference` | real photos labelled with a piece id | yes | photo↔photo, no domain gap — the accurate path |
| `direct` | the piece artwork itself | no | photo↔drawing, distance dominated by rendering style |
| `hybrid` | reference if the category has ≥`MIN_REFS`, else direct | partial | what you will actually run while the ref set is incomplete |

In [ ]:
ENCODER = {
    "eyes":       "arcface",
    "eyebrows":   "arcface",
    "nose":       "arcface",
    "mouth":      "arcface",
    "face_shape": "arcface",
    "hair":       "clip",
    "beard":      "clip",
    "skin_tone":  "skin",
}

MATCH_MODE = "hybrid"
MIN_REFS   = 3          # per category: fewer labelled refs than this and 'reference' mode is noise

def embed(crops_rgb, encoder):
    if not len(crops_rgb):
        return np.zeros((0, 512), np.float32)
    return {"arcface": embed_arcface, "clip": embed_clip, "skin": embed_skin}[encoder](crops_rgb)

def mode_for(category):
    if MATCH_MODE in ("reference", "direct"):
        return MATCH_MODE
    return "reference" if len(REFS.get(category, [])) >= MIN_REFS else "direct"

## Cell 8 — Build the index

One FAISS index **per category**. This is the structural point of the whole design: a global index would let
a beard win an eye query. Searching within a category also means the index is tiny, so `IndexFlatIP` (exact,
brute force) is correct — approximate search buys nothing at this scale and only adds recall error.

Note what happens on `direct` vs `reference` vectors:

- `direct` → the vector is the **piece artwork**, cropped by the same landmark pipeline where a face is
  detectable in it, else centre-cropped (isolated part assets have no face to detect).
- `reference` → the vector is a **real photo**, and its `piece_id` label is carried in the metadata list.
  Several vectors can map to one piece; the search returns a piece, not a photo.

Embeddings are cached to Drive keyed by encoder+mode, so re-running the notebook does not re-encode the bank.

In [ ]:
import faiss, pickle, hashlib
from tqdm.auto import tqdm

def _prep_piece_image(path, out_size=224):
    # Piece assets are usually isolated parts on transparency: flatten onto white
    # (alpha would otherwise read as black and dominate the embedding), then square-pad.
    raw = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if raw is None:
        return None
    if raw.ndim == 3 and raw.shape[2] == 4:
        a = raw[:, :, 3:4].astype(np.float32) / 255.0
        rgb = cv2.cvtColor(raw[:, :, :3], cv2.COLOR_BGR2RGB).astype(np.float32)
        raw = (rgb * a + 255.0 * (1 - a)).astype(np.uint8)
    else:
        raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
    h, w = raw.shape[:2]
    s = max(h, w)
    canvas = np.full((s, s, 3), 255, np.uint8)
    canvas[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = raw
    return cv2.resize(canvas, (out_size, out_size), interpolation=cv2.INTER_CUBIC)

def build_category(category):
    # -> dict(index, ids, vectors, mode, n_failed)
    mode, enc = mode_for(category), ENCODER[category]
    cache = CACHE_DIR / f"{category}__{enc}__{mode}.pkl"
    if cache.exists():
        with open(cache, 'rb') as f:
            d = pickle.load(f)
        d["index"] = _make_index(d["vectors"], enc)
        return d

    ids, crops, failed = [], [], 0
    if mode == "reference":
        for piece_id, path in tqdm(REFS[category], desc=f"{category}:refs"):
            c = crops_from_photo(path, categories=[category])
            if category in c:
                ids.append(piece_id); crops.append(c[category])
            else:
                failed += 1
    else:
        for piece_id, path in tqdm(BANK[category], desc=f"{category}:pieces"):
            img = _prep_piece_image(path)
            if img is None:
                failed += 1; continue
            ids.append(piece_id); crops.append(img)

    vecs = embed(crops, enc) if crops else np.zeros((0, 512), np.float32)
    d = dict(ids=ids, vectors=vecs, mode=mode, encoder=enc, n_failed=failed)
    with open(cache, 'wb') as f:
        pickle.dump(d, f)
    d["index"] = _make_index(vecs, enc)
    return d

def _make_index(vecs, encoder):
    # skin_tone is not a cosine space -> no FAISS index; cell 9 uses deltaE directly.
    if encoder == "skin" or len(vecs) == 0:
        return None
    ix = faiss.IndexFlatIP(vecs.shape[1])   # inner product on L2-normalised vectors == cosine
    ix.add(vecs.astype(np.float32))
    return ix

INDEXES = {}
for cat in CATEGORIES:
    if not BANK[cat]:
        print(f"skip {cat}: no pieces"); continue
    INDEXES[cat] = build_category(cat)
    d = INDEXES[cat]
    print(f"{cat:<12} mode={d['mode']:<10} enc={d['encoder']:<8} "
          f"vectors={len(d['ids']):<5} unique_pieces={len(set(d['ids'])):<5} failed={d['n_failed']}")

## Cell 9 — Query: photo → piece per category

`search_piece` returns the **top-k pieces**, not the top-k vectors. In `reference` mode several vectors share
one `piece_id`, so scores are aggregated per piece before ranking. Aggregation is `max` rather than `mean`
deliberately: one very close reference is stronger evidence than several mediocre ones, and `mean` punishes
a piece for having many diverse reference photos — which is exactly the piece you most want to win.

The returned `score` is cosine ∈ [-1, 1] (higher = closer) except for `skin_tone`, where it is negated ΔE
so that "higher = closer" holds everywhere and the caller does not need a special case.

In [ ]:
from collections import defaultdict

def search_piece(crop_rgb, category, k=5):
    d = INDEXES[category]
    enc = d["encoder"]
    q = embed([crop_rgb], enc)[0]

    if enc == "skin":
        dist = skin_distance(q, d["vectors"])
        order = np.argsort(dist)
        agg = {}
        for i in order:
            agg.setdefault(d["ids"][i], -float(dist[i]))     # negate: higher == better
        return sorted(agg.items(), key=lambda kv: -kv[1])[:k]

    # Over-fetch: with many refs per piece the top-N vectors can all belong to one piece,
    # so fetching only k vectors would return fewer than k distinct pieces.
    n_fetch = min(len(d["ids"]), max(k * 10, 50))
    sims, idxs = d["index"].search(q.reshape(1, -1).astype(np.float32), n_fetch)
    best = defaultdict(lambda: -1e9)
    for s, i in zip(sims[0], idxs[0]):
        if i < 0:
            continue
        pid = d["ids"][i]
        best[pid] = max(best[pid], float(s))
    return sorted(best.items(), key=lambda kv: -kv[1])[:k]

def build_avatar(photo_path, k=3, verbose=True):
    crops = crops_from_photo(photo_path)
    if not crops:
        raise RuntimeError(f"no face detected in {photo_path}")
    result = {}
    for cat in CATEGORIES:
        if cat not in INDEXES or cat not in crops:
            continue
        result[cat] = search_piece(crops[cat], cat, k=k)
    if verbose:
        print(Path(photo_path).name)
        print(f"  {'category':<12}{'top-1':<16}{'score':>8}   runners-up")
        for cat, hits in result.items():
            top, sc = hits[0]
            ru = ", ".join(f"{p}({s:.2f})" for p, s in hits[1:])
            print(f"  {cat:<12}{top:<16}{sc:>8.3f}   {ru}")
    return result, crops

# --- run it ---
_qs = [p for p in sorted(QUERIES_DIR.glob('*')) if p.suffix.lower() in IMG_EXT]
if _qs:
    SELECTION, QCROPS = build_avatar(_qs[0])
else:
    print("no query images found in", QUERIES_DIR)

## Cell 10 — Measure it (leave-one-out A/B)

This is the cell that turns the architecture from a plausible design into a measured one, and the reason
to insist on it is in the doc's own closing caveat: nothing here has been validated on real data.

Protocol: for every reference photo, embed it, remove **that photo** from the index (not the whole piece),
search, and check whether the true `piece_id` comes back at rank 1 and within rank 3. Repeat with each
encoder. Requires ≥2 refs per piece for the leave-one-out to be meaningful — with 1 ref per piece, removing
it removes the only correct answer and accuracy is 0 by construction.

Read the output like this:

- **ArcFace ≫ CLIP on a category** → the doc's routing was right for it.
- **CLIP ≥ ArcFace** → flip that entry in `ENCODER`; ArcFace's identity-only training discarded the signal.
- **Both near random** (`1/n_pieces`) → the problem is the crop or the labels, not the encoder. Go back to cell 5.

In [ ]:
def loo_eval(category, encoder, max_n=400):
    items = REFS.get(category, [])
    ids_all = [i for i, _ in items]
    counts = defaultdict(int)
    for i in ids_all:
        counts[i] += 1
    usable = [(i, p) for i, p in items if counts[i] >= 2]
    if len(usable) < 5:
        return None
    usable = usable[:max_n]

    crops, ids = [], []
    for pid, path in usable:
        c = crops_from_photo(path, categories=[category])
        if category in c:
            crops.append(c[category]); ids.append(pid)
    if len(crops) < 5:
        return None

    V = embed(crops, encoder)
    ids = np.array(ids)

    if encoder == "skin":
        D = np.array([skin_distance(v, V) for v in V])
        S = -D
    else:
        S = V @ V.T
    np.fill_diagonal(S, -1e9)                       # leave-one-out: a photo cannot retrieve itself

    top1 = top3 = 0
    for r in range(len(ids)):
        order = np.argsort(-S[r])
        seen, ranked = set(), []
        for j in order:                              # collapse vectors -> distinct pieces
            if ids[j] not in seen:
                seen.add(ids[j]); ranked.append(ids[j])
            if len(ranked) == 3:
                break
        top1 += ranked[0] == ids[r]
        top3 += ids[r] in ranked
    n = len(ids)
    return dict(n=n, n_pieces=len(set(ids)), top1=top1 / n, top3=top3 / n,
                chance=1 / len(set(ids)))

rows = []
for cat in CATEGORIES:
    for enc in ["arcface", "clip"] + (["skin"] if cat == "skin_tone" else []):
        r = loo_eval(cat, enc)
        if r:
            rows.append((cat, enc, r))

print(f"{'category':<12}{'encoder':<10}{'n':>5}{'pieces':>8}{'top1':>8}{'top3':>8}{'chance':>8}  verdict")
best = {}
for cat, enc, r in rows:
    print(f"{cat:<12}{enc:<10}{r['n']:>5}{r['n_pieces']:>8}{r['top1']:>8.2f}{r['top3']:>8.2f}"
          f"{r['chance']:>8.2f}  {'above chance' if r['top1'] > 2*r['chance'] else 'NOT above chance'}")
    if r['top1'] > best.get(cat, (None, -1))[1]:
        best[cat] = (enc, r['top1'])

if best:
    print("\nmeasured routing (paste into ENCODER in cell 7 if it differs):")
    for cat, (enc, acc) in best.items():
        flag = "" if enc == ENCODER[cat] else f"   <-- CHANGED from {ENCODER[cat]}"
        print(f'    "{cat}": "{enc}",   # top1={acc:.2f}{flag}')
else:
    print("\nnot enough labelled refs (need >=2 photos per piece) to evaluate anything yet.")

## Cell 11 — Composite preview

Honest limitation: correct layering needs an **anchor convention** in your assets — each PNG rendered on a
common canvas at a common scale, or shipped with an anchor point. Without that, no amount of code places an
eye correctly, because "where does this eye go" is not recoverable from the pixels.

Two paths:

- `LAYOUT = "canvas"` — assets are pre-registered full-canvas PNGs with transparency. Compositing is then a
  plain alpha stack in `CATEGORIES` order. This is what most avatar asset packs do and it is the one to aim for.
- `LAYOUT = "anchored"` — assets are tight crops; the notebook places them at the landmark centroid of the
  corresponding region on a reference head, scaled by IOD. Approximate, useful for eyeballing results only.

In [ ]:
LAYOUT = "canvas"
CANVAS = 512

def _rgba(path, size=None):
    raw = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if raw is None:
        return None
    if raw.ndim == 2:
        raw = cv2.cvtColor(raw, cv2.COLOR_GRAY2BGRA)
    if raw.shape[2] == 3:
        raw = cv2.cvtColor(raw, cv2.COLOR_BGR2BGRA)
    rgba = np.dstack([cv2.cvtColor(raw[:, :, :3], cv2.COLOR_BGR2RGB), raw[:, :, 3]])
    if size:
        rgba = cv2.resize(rgba, size, interpolation=cv2.INTER_AREA)
    return rgba

def _alpha_over(dst, src):
    a = src[:, :, 3:4].astype(np.float32) / 255.0
    dst[:, :, :3] = (src[:, :, :3] * a + dst[:, :, :3] * (1 - a)).astype(np.uint8)
    dst[:, :, 3:4] = np.maximum(dst[:, :, 3:4], src[:, :, 3:4])
    return dst

def composite(selection):
    path_of = {cat: dict(BANK[cat]) for cat in CATEGORIES}
    out = np.zeros((CANVAS, CANVAS, 4), np.uint8)
    for cat in CATEGORIES:                      # CATEGORIES order == z-order
        if cat not in selection or cat == "skin_tone":
            continue
        pid = selection[cat][0][0]
        p = path_of[cat].get(pid)
        if p is None:
            continue
        layer = _rgba(p, (CANVAS, CANVAS)) if LAYOUT == "canvas" else _rgba(p)
        if layer is None:
            continue
        if LAYOUT == "canvas":
            out = _alpha_over(out, layer)
        else:
            h, w = layer.shape[:2]
            y, x = CANVAS // 2 - h // 2, CANVAS // 2 - w // 2   # placeholder placement
            region = out[y:y+h, x:x+w]
            out[y:y+h, x:x+w] = _alpha_over(region, layer)
    return out

if _qs:
    avatar = composite(SELECTION)
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(load_rgb(_qs[0])); ax[0].set_title("input"); ax[0].axis('off')
    ax[1].imshow(avatar);           ax[1].set_title("assembled"); ax[1].axis('off')
    plt.tight_layout(); plt.show()

## Cell 12 — Persist

Saves indices + id lists + the active config to Drive. The config is saved *with* the vectors on purpose:
an embedding file is meaningless without knowing which encoder, crop padding and match mode produced it,
and mixing two generations of vectors in one index is a failure that produces plausible-looking wrong
answers rather than an error.

**Rebuild triggers** — anything that changes the query-side transform invalidates the bank-side vectors:
changing `CROP_CFG`, changing `ENCODER`, changing `MATCH_MODE`, or swapping the CLIP checkpoint. Adding a
new *piece* does not; that only appends.

In [ ]:
import json, shutil

OUT = ROOT / 'artifacts'
OUT.mkdir(exist_ok=True)

manifest = dict(
    categories=CATEGORIES, encoder=ENCODER, match_mode=MATCH_MODE,
    min_refs=MIN_REFS, crop_cfg=CROP_CFG,
    clip_checkpoint='ViT-B-32/laion2b_s34b_b79k', arcface='buffalo_l/w600k_r50',
    counts={c: dict(pieces=len(BANK[c]), refs=len(REFS[c])) for c in CATEGORIES},
)
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))

for cat, d in INDEXES.items():
    np.save(OUT / f'{cat}.vectors.npy', d['vectors'])
    (OUT / f'{cat}.ids.json').write_text(json.dumps(d['ids']))
    if d['index'] is not None:
        faiss.write_index(d['index'], str(OUT / f'{cat}.faiss'))

print("written to", OUT)
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:<28}{p.stat().st_size/1024:>10.1f} KB")

## What to do next, in order

1. Run cells 1–5 and **look at the crops**. Fix `CROP_CFG` before anything else.
2. Run cell 8 with `MATCH_MODE = "direct"` to get a baseline, then cell 9 on 5 photos. Expect this to look mediocre — that is the domain gap, not a bug.
3. Load the real-people images into `refs/<category>/<piece_id>/` and run cell 10. Two refs per piece is the minimum for a number; five is where it starts to mean something.
4. Set `ENCODER` from cell 10's measured routing, not from the doc's assumption.
5. Only then port to FastAPI. The Colab is the measurement rig; the service is a thin wrapper around `crops_from_photo` + `search_piece` with the artifacts from cell 12 loaded at startup.

**Open question for the FastAPI port:** does each category get an independent argmax (current behaviour), or
does the avatar need global coherence — e.g. hair colour constraining eyebrow colour? Independent argmax is
simpler and is what the doc describes; joint selection is a different and considerably harder problem.